# Train the glyph segmentation detector (Colab, GPU)

Run this notebook on Colab with a GPU runtime
(Runtime -> Change runtime type -> GPU).

This fine-tunes a pretrained YOLOv8-segmentation checkpoint (see
`reports/2026-09-15-segmentation-detector-sourcing.md`) on synthetic
composite images built from our own single-glyph crops
(`data/raw/`) -- see `docs/superpowers/specs/2026-09-15-glyph-segmentation-detector-design.md`
for the full design.

## 1. Clone the repo and install it

In [ ]:
!git clone https://github.com/DelfinEryilmaz/hieroglyph-translator.git
%cd hieroglyph-translator
!pip install -e . -q
!pip install ultralytics -q

## 2. Mount Google Drive (so training survives a disconnect)

Colab's local disk is wiped every time this runtime disconnects or resets
(idle timeout, a lost connection, or the ~12-hour hard limit) -- so a
partially- or fully-trained checkpoint saved only to local disk would be
lost with it. Mounting Drive and pointing training's output directory
there means every epoch's checkpoint lands somewhere that survives a
disconnect. If this runtime does disconnect mid-training, reconnect,
re-run the setup cells above (cloning and installing again is quick), and
check `DRIVE_RUNS_DIR` in Drive for whatever the last completed epoch
saved -- training itself will need to be started again from this notebook,
but nothing already-saved is lost.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_RUNS_DIR = "/content/drive/MyDrive/hieroglyph-detector-runs"
print(f"Training checkpoints will be saved under: {DRIVE_RUNS_DIR}")

## 3. Check GPU is available

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type > GPU")

## 4. Upload the glyph crop dataset

Same dataset the classifier trains on. Upload the original `archive.zip`
from Kaggle (or a zip of your local `data/raw/` folder) -- either works,
this cell handles both.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick archive.zip (from Kaggle) or your own data_raw.zip
uploaded_filename = next(iter(uploaded))

In [ ]:
import shutil
import zipfile
from pathlib import Path

DATA_ROOT = Path("data/raw")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(uploaded_filename) as zf:
    zf.extractall(DATA_ROOT)


def class_folders(root):
    return [p for p in root.iterdir() if p.is_dir()]


top_level = class_folders(DATA_ROOT)
# If the zip had one extra wrapper folder (some Windows re-zips do this),
# flatten it so DATA_ROOT/<class>/*.png is the actual layout.
if len(top_level) == 1 and not list(top_level[0].glob("*.png")):
    inner = top_level[0]
    for child in inner.iterdir():
        shutil.move(str(child), str(DATA_ROOT / child.name))
    inner.rmdir()

print(f"{len(class_folders(DATA_ROOT))} class folders under {DATA_ROOT}")

## 5. Split crop files, then generate synthetic composites

Splitting happens on individual *crop files* before compositing, not on
composites -- so no single glyph crop's pixels ever appear in both the
train and test splits (see `hieroglyph.segmentation.synthesize.split_crop_paths`'s
docstring).

Uses `generate_mixed_dataset`, which mixes dense/column composites (modeling
tightly-packed real papyrus text) with the original sparse/scatter
composites (modeling sparser wall-carving-style photos), so the detector
sees both real-photo layouts during training -- see
`docs/superpowers/specs/2026-09-17-dense-text-detection-design.md`.

In [ ]:
from pathlib import Path
from hieroglyph.segmentation.synthesize import generate_mixed_dataset, list_crop_paths, split_crop_paths

RAW_DIR = Path("data/raw")
SYNTH_DIR = Path("data/synthetic_composites")

crop_paths = list_crop_paths(RAW_DIR)
train_crops, valid_crops, test_crops = split_crop_paths(crop_paths, seed=0)
print(f"{len(train_crops)} train crops, {len(valid_crops)} valid crops, {len(test_crops)} test crops")

generate_mixed_dataset(train_crops, SYNTH_DIR / "train", num_composites=4000, seed=0)
generate_mixed_dataset(valid_crops, SYNTH_DIR / "valid", num_composites=500, seed=1)
generate_mixed_dataset(test_crops, SYNTH_DIR / "test", num_composites=500, seed=2)
print("done generating composites (mix of dense/column and sparse/scatter)")

In [ ]:
import yaml

data_yaml = {
    "path": str(SYNTH_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {0: "hieroglyph"},
    "nc": 1,
}
(SYNTH_DIR / "data.yaml").write_text(yaml.dump(data_yaml), encoding="utf-8")
print((SYNTH_DIR / "data.yaml").read_text())

## 6. Download the pretrained checkpoint we're fine-tuning from

In [ ]:
!python scripts/download_pretrained_yolo.py --dest models/yolo_seg_pretrained.pt

## 7. Fine-tune

In [ ]:
from ultralytics import YOLO

model = YOLO("models/yolo_seg_pretrained.pt")
train_results = model.train(
    data=str(SYNTH_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    device=0,
    project=DRIVE_RUNS_DIR,
    name="yolo_seg_finetune",
)

## 8. Fine-tune further on real annotated photos

Synthetic composites (steps 5-7) teach volume and glyph shape, but no
synthesizer fully reproduces real ink texture, lighting, or surface noise
-- this stage continues training the *same* `model` object (its current
weights, not a fresh checkpoint) for a short run on
`data/real_eval_photos/{train,valid}` (83 + 7 real, human-annotated photo
files, committed to the repo -- see
`reports/2026-09-18-real-finetune-data-sourcing.md`), remapped from their
original 19-Gardiner-class detection labels down to our single
"hieroglyph" class. See
`docs/superpowers/specs/2026-09-18-real-data-finetune-and-hard-negatives-design.md`
for why this small-but-real stage is expected to help where more
synthetic data alone can't.

### What this data actually is (measured -- read before trusting the result)

These are **single-glyph crops, not multi-sign scene photos.** Measured
over all 90 committed image/label pairs:

- 90 images, **90 boxes total -- exactly one box per image.**
- Median box area is **39.5% of the 640x640 frame**, and **40% of boxes
  cover more than half the frame** (min 12.8%, max 99.2%).
- The 83 train files are really **28 distinct source photos** (27 with
  three Roboflow rotation-augmented copies each, one with two); valid is 7
  distinct photos with no source-photo overlap. So the effective distinct
  sample size is **35 photos, not 90**, even though 83 + 7 is the right
  file count to feed training.
- Many are visibly upscaled from small sources (blocky), with black
  letterboxing corners left by Roboflow's rotation augmentation.

So understand this stage as *"expose the model to real photographic
texture and noise on individual signs"* -- **not** *"expose it to
realistic real-world sign density and layout."* Dense, realistic layout is
what the dense column composites and hard-negative composites of steps 5-7
are for. Training 15 epochs on one-huge-sign-per-frame data can plausibly
push the model *away* from densely-packed-text performance, which is
exactly why step 9 has you evaluate this run's checkpoint against step 7's
rather than assuming it wins.

In [ ]:
from hieroglyph.segmentation.real_data import prepare_real_finetune_split

REAL_DIR = Path("data/real_eval_photos")
REAL_FINETUNE_DIR = Path("data/real_finetune")

train_count = prepare_real_finetune_split(
    REAL_DIR / "train" / "images", REAL_DIR / "train" / "labels", REAL_FINETUNE_DIR / "train"
)
valid_count = prepare_real_finetune_split(
    REAL_DIR / "valid" / "images", REAL_DIR / "valid" / "labels", REAL_FINETUNE_DIR / "valid"
)
print(f"{train_count} real train pairs, {valid_count} real valid pairs -> {REAL_FINETUNE_DIR}")

In [ ]:
real_data_yaml = {
    "path": str(REAL_FINETUNE_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "names": {0: "hieroglyph"},
    "nc": 1,
}
(REAL_FINETUNE_DIR / "data.yaml").write_text(yaml.dump(real_data_yaml), encoding="utf-8")
print((REAL_FINETUNE_DIR / "data.yaml").read_text())

In [ ]:
# Continues from the synthetic-pretrain weights already in `model` (calling
# .train() again on the same YOLO instance resumes from its current
# in-memory weights) -- a short, low-volume fine-tune, not a from-scratch
# run on 90 images.
real_train_results = model.train(
    data=str(REAL_FINETUNE_DIR / "data.yaml"),
    epochs=15,
    imgsz=640,
    device=0,
    project=DRIVE_RUNS_DIR,
    name="yolo_seg_finetune_real",
)

## 9. Download both candidate checkpoints, then pick the better one

Both runs' checkpoints are already saved in Google Drive (`DRIVE_RUNS_DIR`,
mounted in step 2) the moment each run finishes, so they already survived
this session -- the cell below just also pulls copies to your local machine
for convenience. There are **two candidates**, and which one is better is a
question to settle by measurement, not by assumption:

- `yolo_seg_synthetic_only.pt` -- step 7's synthetic-only run
  (`yolo_seg_finetune`), trained on the synthetic composites including the
  dense-column and hard-negative ones.
- `yolo_seg_real_finetuned.pt` -- step 8's real-photo fine-tune
  (`yolo_seg_finetune_real`): those same weights after 15 further epochs on
  the 90 real photo files (35 distinct source photos -- see step 8).

The real-fine-tuned checkpoint is **not** automatically the winner. As
step 8's note records, that data is 90 single-glyph crops (one box per
image, median box area ~40% of the frame), so those 15 epochs may well have
traded away dense-text performance for photographic realism on isolated
signs.

**Evaluate both** with `notebooks/05_evaluate_segmenter.ipynb` -- especially
its Section 5 real-papyrus qualitative check (box count and placement on a
real, densely-written papyrus photo), alongside the quantitative metrics on
the synthetic test split. Then copy whichever one actually performs better
to `models/yolo_seg.pt`: that exact filename is what
`notebooks/05_evaluate_segmenter.ipynb` and the Streamlit demo expect.

In [ ]:
import shutil

from google.colab import files

# Both runs' weights files are named best.pt inside their own run directory,
# so copy each to a distinct local name first -- otherwise the two downloads
# collide and you can't tell which checkpoint is which.
CANDIDATE_CHECKPOINTS = {
    "yolo_seg_synthetic_only.pt": f"{DRIVE_RUNS_DIR}/yolo_seg_finetune/weights/best.pt",
    "yolo_seg_real_finetuned.pt": f"{DRIVE_RUNS_DIR}/yolo_seg_finetune_real/weights/best.pt",
}

for local_name, run_weights in CANDIDATE_CHECKPOINTS.items():
    shutil.copy(run_weights, local_name)
    print(f"{run_weights} -> {local_name}")
    files.download(local_name)

print(
    "\nEvaluate BOTH with notebooks/05_evaluate_segmenter.ipynb (especially "
    "Section 5's real-papyrus check), then copy the better one to models/yolo_seg.pt"
)